# 04 - Statistical Analysis

## RHoMIS Analytics — Section C, Group 7

**Notebook goal**

This notebook tests whether the descriptive patterns from notebook 03 are statistically supported and identifies the most decision-useful drivers of household food vulnerability.

**Plain-English role of this notebook**

Notebook 04 is the evidence notebook. Notebook 03 showed patterns; this notebook checks whether those patterns are strong enough to use in the dashboard and final recommendations.

**Statistical focus**

- validate headline EDA patterns with formal tests
- separate strong drivers from weak or misleading ones
- estimate a multivariable vulnerability model
- build interpretable farm-household profiles for Tableau

**Methodological guardrails**

- `foodshortagetime` remains the main full-dataset outcome
- FIES and HFIAS are used for validation, not as full-dataset replacements
- income uses PPP-normalized indicator fields, not pooled raw local currency
- results are interpreted as association, not causation

## What This Notebook Is Doing

To keep the analysis explainable, the notebook uses four layers only:

| Layer | Method | Why it is included |
|---|---|---|
| 1 | Chi-square tests | Checks whether categorical drivers like irrigation are linked to food shortage |
| 2 | Mann-Whitney and Spearman tests | Handles skewed numeric variables such as land and productivity without assuming normal data |
| 3 | Logistic regression | Checks which drivers still matter after controlling for country and year |
| 4 | K-means profiles | Creates a small number of household vulnerability profiles for Tableau drill-down |

The model is not used to claim causation. It is used to avoid weak dashboard claims such as "crop diversity is the main driver" when that pattern weakens after controls.

## 1. Setup

The runtime environment for this notebook needs the scientific stack used for testing and modeling: `scipy`, `statsmodels`, `scikit-learn`, `matplotlib`, and `seaborn`.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
    from scipy import stats
    from sklearn.cluster import KMeans
    from sklearn.metrics import roc_auc_score, silhouette_score
    from sklearn.preprocessing import StandardScaler
    import statsmodels.formula.api as smf
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook 04 requires matplotlib, seaborn, scipy, scikit-learn, and statsmodels in the notebook environment."
    ) from exc

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="talk")
sns.set_palette("deep")

### 1.1 Load the cleaned dataset and indicator layer

This block recreates the full statistical analysis input by combining the cleaned ETL output with the household-level indicator file used throughout the analysis phase.

In [2]:
data_candidates = [
    Path("../data/processed/rhomis_cleaned.csv.gz"),
    Path("data/processed/rhomis_cleaned.csv.gz"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
assert DATA_PATH is not None, f"Missing cleaned dataset. Checked: {data_candidates}"

df = pd.read_csv(DATA_PATH, compression="gzip")
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

indicator_candidates = [
    Path(".ai-workflow/research/indicator_data.tab"),
    Path("../.ai-workflow/research/indicator_data.tab"),
    Path("data/raw/indicator_data.tab"),
    Path("../data/raw/indicator_data.tab"),
]
INDICATOR_PATH = next((path for path in indicator_candidates if path.exists()), None)
assert INDICATOR_PATH is not None, (
    "Missing indicator_data.tab. Keep the household indicator file in .ai-workflow/research/ "
    "or data/raw/ before rerunning this notebook."
)

indicator_usecols = [
    "id_unique",
    "hh_size_mae",
    "livestock_tlu",
    "nr_months_food_shortage",
    "hfias_status",
    "fies_score",
    "hdds_good_season",
    "hdds_bad_season",
    "hdds_last_month",
    "total_income_lcu_per_year",
    "currency_conversion_lcu_to_ppp",
    "value_farm_products_consumed_lcu_per_hh_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
]
indicators = pd.read_csv(INDICATOR_PATH, sep="\t", usecols=indicator_usecols)
indicators = indicators.rename(columns={"fies_score": "fies_score_indicator", "hfias_status": "hfias_status_indicator"})
df = df.merge(indicators, on="id_unique", how="left", validate="one_to_one")
print(f"Merged household indicator data: {indicators.shape[1] - 1} fields")

Loaded cleaned dataset: 54,873 rows x 77 columns
Merged household indicator data: 12 fields


## 2. Recreate the Verified Analysis Layer

Notebook 04 rebuilds the approved analysis fields so that the statistics are fully reproducible from the cleaned ETL output.

In [3]:
fies_cols = [f"fies_{i}" for i in range(1, 9)]
hfias_cols = [f"hfias_{i}" for i in range(1, 10)]
crop_name_cols = [f"crop_name_{i}" for i in range(1, 6)]
harvest_cols = [f"crop_harvest_kg_per_year_{i}" for i in range(1, 4)]
income_cols = [f"crop_income_per_year_{i}" for i in range(1, 4)] + [f"livestock_sale_income_{i}" for i in range(1, 3)]

def count_distinct_non_null(values) -> int:
    cleaned = {
        str(value).strip().lower()
        for value in values
        if pd.notna(value) and str(value).strip() != ""
    }
    return len(cleaned)

hfias_frequency_map = {
    "never": 0,
    "monthly": 1,
    "fewpermonth": 2,
    "weekly": 3,
    "fewperweek": 4,
    "daily": 5,
}

low_education_levels = {"no_school", "illiterate", "koranic"}

df["food_shortage_flag"] = np.where(
    df["foodshortagetime"].isin(["y", "n"]),
    (df["foodshortagetime"] == "y").astype(int),
    np.nan,
)
crop_name_diversity_count = df[crop_name_cols].apply(count_distinct_non_null, axis=1)
crop_count_numeric = pd.to_numeric(df["crop_count"], errors="coerce")
df["crop_diversity_count"] = crop_count_numeric.combine_first(crop_name_diversity_count).clip(lower=0)
df["crop_harvest_total_kg_observed"] = (
    df[harvest_cols]
    .apply(pd.to_numeric, errors="coerce")
    .where(df[harvest_cols].apply(pd.to_numeric, errors="coerce") > 0)
    .sum(axis=1, min_count=1)
)
df["land_productivity_kg_per_ha"] = np.where(
    (df["crop_harvest_total_kg_observed"] > 0) & (df["land_cultivated_ha"] > 0),
    df["crop_harvest_total_kg_observed"] / df["land_cultivated_ha"],
    np.nan,
)
df["farm_cash_income_lcu_observed"] = df[income_cols].sum(axis=1, min_count=1)
df["total_income_ppp_per_year"] = np.where(
    (df["total_income_lcu_per_year"].notna()) & (df["currency_conversion_lcu_to_ppp"] > 0),
    df["total_income_lcu_per_year"] / df["currency_conversion_lcu_to_ppp"],
    np.nan,
)
df["income_ppp_per_mae"] = np.where(
    (df["total_income_ppp_per_year"].notna()) & (df["hh_size_mae"] > 0),
    df["total_income_ppp_per_year"] / df["hh_size_mae"],
    np.nan,
)
df["country_income_rank_pct"] = df.groupby("country")["total_income_ppp_per_year"].rank(pct=True)
df["irrigated_flag"] = np.where(
    df["land_irrigated"].isin(["y", "n"]),
    (df["land_irrigated"] == "y").astype(int),
    np.nan,
)
df["offfarm_flag"] = np.where(
    df["offfarm_incomes_any"].isin(["y", "n"]),
    (df["offfarm_incomes_any"] == "y").astype(int),
    np.nan,
)
df["female_respondent_flag"] = np.where(
    df["respondentsex"].isin(["f", "m"]),
    (df["respondentsex"] == "f").astype(int),
    np.nan,
)
df["low_education_flag"] = np.where(
    df["education_level"].notna(),
    df["education_level"].isin(low_education_levels).astype(int),
    np.nan,
)
df["fies_complete"] = df[fies_cols].notna().all(axis=1)
fies_count_from_module = np.where(df["fies_complete"], df[fies_cols].eq("y").sum(axis=1), np.nan)
df["fies_yes_count"] = df["fies_score_indicator"].combine_first(pd.Series(fies_count_from_module, index=df.index))
df["hfias_complete"] = df[hfias_cols].notna().all(axis=1)
df["hfias_frequency_score"] = np.where(
    df["hfias_complete"],
    df[hfias_cols].apply(lambda row: sum(hfias_frequency_map.get(value, np.nan) for value in row), axis=1),
    np.nan,
)

total_rows = len(df)
print(f"Food-shortage analysis population: {int(df['food_shortage_flag'].notna().sum()):,}")
print(f"Model-safe within-country income rank coverage: {int(df['country_income_rank_pct'].notna().sum()):,}")

Food-shortage analysis population: 47,399
Model-safe within-country income rank coverage: 50,964


## 3. Statistical Question Set

This notebook answers four questions:

1. Are the strongest EDA patterns statistically reliable?
2. Which structural variables still matter after controlling for country and year?
3. Which attractive-looking variables weaken after controls?
4. Can we define interpretable vulnerability profiles for Tableau?

In simple terms: notebook 04 decides what deserves to move from "interesting chart" to "dashboard/recommendation evidence."

## 4. Hypothesis Tests for Core Categorical Drivers

We use chi-square tests for categorical features and report both the p-value and Cramér's V, so significance is not confused with practical strength.

In [4]:
def cramers_v(confusion: pd.DataFrame) -> float:
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    return np.sqrt((chi2 / n) / max(min(r - 1, k - 1), 1))

categorical_tests = [
    ("land_irrigated", "Irrigation access"),
    ("offfarm_incomes_any", "Off-farm participation"),
    ("respondentsex", "Respondent sex"),
    ("low_education_flag", "Low education proxy"),
]

chi_square_results = []
for column, label in categorical_tests:
    sub = df[df["food_shortage_flag"].notna() & df[column].notna()].copy()
    table = pd.crosstab(sub[column], sub["food_shortage_flag"])
    chi2, p_value, dof, _ = stats.chi2_contingency(table)
    chi_square_results.append(
        {
            "driver": label,
            "column": column,
            "valid_rows": int(sub.shape[0]),
            "chi2": round(float(chi2), 3),
            "p_value": float(p_value),
            "cramers_v": round(float(cramers_v(table)), 3),
        }
    )

chi_square_results = pd.DataFrame(chi_square_results).sort_values("p_value")
display(chi_square_results)

,driver,column,valid_rows,chi2,p_value,cramers_v
3,Low education proxy,low_education_flag,41533,874.103,4.183609e-192,0.145
0,Irrigation access,land_irrigated,45353,869.518,4.153403e-191,0.138
2,Respondent sex,respondentsex,45710,281.520,3.502519e-63,0.078
1,Off-farm participation,offfarm_incomes_any,47351,152.198,5.734926e-35,0.057


## 5. Robust Numeric Group Comparisons

Numeric variables in this dataset are highly skewed, so we avoid naive t-tests and use Mann-Whitney U tests plus a rank-biserial effect size.

In [5]:
def rank_biserial_from_u(u_stat: float, n1: int, n2: int) -> float:
    return (2 * u_stat) / (n1 * n2) - 1

numeric_tests = [
    ("household_size_derived", "Household size"),
    ("land_cultivated_ha", "Cultivated land (ha)"),
    ("crop_diversity_count", "Crop diversity count"),
    ("land_productivity_kg_per_ha", "Land productivity (kg/ha)"),
    ("country_income_rank_pct", "Within-country income rank"),
]

mann_whitney_results = []
for column, label in numeric_tests:
    sub = df[df["food_shortage_flag"].notna() & df[column].notna()].copy()
    shortage = sub.loc[sub["food_shortage_flag"] == 1, column]
    no_shortage = sub.loc[sub["food_shortage_flag"] == 0, column]
    u_stat, p_value = stats.mannwhitneyu(shortage, no_shortage, alternative="two-sided")
    mann_whitney_results.append(
        {
            "driver": label,
            "column": column,
            "valid_rows": int(sub.shape[0]),
            "median_shortage": float(shortage.median()),
            "median_no_shortage": float(no_shortage.median()),
            "p_value": float(p_value),
            "rank_biserial": round(float(rank_biserial_from_u(u_stat, len(shortage), len(no_shortage))), 3),
        }
    )

mann_whitney_results = pd.DataFrame(mann_whitney_results).sort_values("p_value")
display(mann_whitney_results)

,driver,column,valid_rows,median_shortage,median_no_shortage,p_value,rank_biserial
3,Land productivity (kg/ha),land_productivity_kg_per_ha,42919,450.000000,1001.000000,0.000000e+00,-0.298
4,Within-country income rank,country_income_rank_pct,45705,0.442754,0.600930,1.010406e-224,-0.185
0,Household size,household_size_derived,47399,6.000000,5.000000,2.075289e-116,0.129
2,Crop diversity count,crop_diversity_count,47399,2.000000,2.000000,2.935365e-06,-0.026
1,Cultivated land (ha),land_cultivated_ha,45053,1.747000,1.618744,2.037544e-02,-0.014


## 6. Correlation Check

Spearman's rho helps identify monotonic relationships without assuming normality.

In [6]:
spearman_results = []
for column, label in numeric_tests:
    sub = df[["food_shortage_flag", column]].dropna()
    rho, p_value = stats.spearmanr(sub["food_shortage_flag"], sub[column])
    spearman_results.append(
        {
            "driver": label,
            "column": column,
            "valid_rows": int(sub.shape[0]),
            "spearman_rho": round(float(rho), 3),
            "p_value": float(p_value),
        }
    )

spearman_results = pd.DataFrame(spearman_results).sort_values("p_value")
display(spearman_results)

,driver,column,valid_rows,spearman_rho,p_value
3,Land productivity (kg/ha),land_productivity_kg_per_ha,42919,-0.240,0.000000e+00
4,Within-country income rank,country_income_rank_pct,45705,-0.150,3.018862e-227
0,Household size,household_size_derived,47399,0.105,4.796888e-117
2,Crop diversity count,crop_diversity_count,47399,-0.021,2.928714e-06
1,Cultivated land (ha),land_cultivated_ha,45053,-0.011,2.037376e-02


## 7. Food-Security Module Validation

Notebook 03 showed that FIES and HFIAS support the main storyline. Here we quantify that support statistically.

In [7]:
fies_validation = df[df["food_shortage_flag"].notna() & df["fies_yes_count"].notna()].copy()
fies_rho, fies_p = stats.spearmanr(fies_validation["food_shortage_flag"], fies_validation["fies_yes_count"])

hfias_validation = df[df["food_shortage_flag"].notna() & df["hfias_frequency_score"].notna()].copy()
hfias_shortage = hfias_validation.loc[hfias_validation["food_shortage_flag"] == 1, "hfias_frequency_score"]
hfias_no_shortage = hfias_validation.loc[hfias_validation["food_shortage_flag"] == 0, "hfias_frequency_score"]
hfias_u, hfias_p = stats.mannwhitneyu(hfias_shortage, hfias_no_shortage, alternative="two-sided")

module_validation = pd.DataFrame(
    [
        {
            "validation_layer": "FIES yes-count vs food shortage",
            "test": "Spearman correlation",
            "valid_rows": int(fies_validation.shape[0]),
            "effect": round(float(fies_rho), 3),
            "p_value": float(fies_p),
        },
        {
            "validation_layer": "HFIAS score by food shortage group",
            "test": "Mann-Whitney U",
            "valid_rows": int(hfias_validation.shape[0]),
            "effect": round(float(rank_biserial_from_u(hfias_u, len(hfias_shortage), len(hfias_no_shortage))), 3),
            "p_value": float(hfias_p),
        },
    ]
)
display(module_validation)

,validation_layer,test,valid_rows,effect,p_value
0,FIES yes-count vs food shortage,Spearman correlation,24097,0.587,0.000000e+00
1,HFIAS score by food shortage group,Mann-Whitney U,5911,0.334,1.312447e-170


## 8. Multivariable Vulnerability Model

The main model is a country- and year-controlled logistic regression with food shortage as the dependent variable.

This is the most important step in the notebook because it tells us which drivers remain important after adjusting for the broader context.

How to explain it:

> We are not predicting individual households for operational deployment. We are estimating which factors are associated with food shortage after accounting for country and survey year differences.

In [8]:
model_df = df[
    [
        "food_shortage_flag",
        "land_productivity_kg_per_ha",
        "land_cultivated_ha",
        "household_size_derived",
        "crop_diversity_count",
        "country_income_rank_pct",
        "irrigated_flag",
        "offfarm_flag",
        "female_respondent_flag",
        "low_education_flag",
        "year",
        "country",
    ]
].dropna().copy()

model_df = model_df[
    (model_df["land_productivity_kg_per_ha"] > 0) & (model_df["land_cultivated_ha"] > 0)
].copy()
model_df["log_land_productivity"] = np.log1p(model_df["land_productivity_kg_per_ha"])
model_df["log_land_cultivated"] = np.log1p(model_df["land_cultivated_ha"])

formula = (
    "food_shortage_flag ~ log_land_productivity + log_land_cultivated + household_size_derived "
    "+ crop_diversity_count + country_income_rank_pct + irrigated_flag + offfarm_flag "
    "+ female_respondent_flag + low_education_flag + C(year) + C(country)"
)

logit_model = smf.logit(formula, data=model_df).fit(disp=0, maxiter=200)
model_auc = roc_auc_score(model_df["food_shortage_flag"], logit_model.predict(model_df))

core_terms = [
    "log_land_productivity",
    "log_land_cultivated",
    "household_size_derived",
    "crop_diversity_count",
    "country_income_rank_pct",
    "irrigated_flag",
    "offfarm_flag",
    "female_respondent_flag",
    "low_education_flag",
]
conf = logit_model.conf_int().loc[core_terms]
coefficient_table = pd.DataFrame(
    {
        "term": core_terms,
        "odds_ratio": np.exp(logit_model.params.loc[core_terms]),
        "ci_low": np.exp(conf[0]),
        "ci_high": np.exp(conf[1]),
        "p_value": logit_model.pvalues.loc[core_terms],
    }
).sort_values("p_value")
coefficient_table[["odds_ratio", "ci_low", "ci_high"]] = coefficient_table[
    ["odds_ratio", "ci_low", "ci_high"]
].round(3)
display(coefficient_table)

model_quality = pd.DataFrame(
    [
        {
            "model_rows": int(model_df.shape[0]),
            "mcfadden_pseudo_r2": round(float(logit_model.prsquared), 3),
            "roc_auc": round(float(model_auc), 3),
        }
    ]
)
display(model_quality)

,term,odds_ratio,ci_low,ci_high,p_value
log_land_productivity,log_land_productivity,0.828,0.810,0.845,4.286681e-70
log_land_cultivated,log_land_cultivated,0.671,0.637,0.706,2.338924e-52
country_income_rank_pct,country_income_rank_pct,0.456,0.408,0.510,4.789568e-43
low_education_flag,low_education_flag,1.268,1.184,1.358,1.093502e-11
household_size_derived,household_size_derived,1.011,1.006,1.017,8.603251e-06
crop_diversity_count,crop_diversity_count,1.032,1.006,1.059,1.649311e-02
irrigated_flag,irrigated_flag,0.926,0.858,0.999,4.792101e-02
offfarm_flag,offfarm_flag,1.036,0.977,1.099,2.324213e-01
female_respondent_flag,female_respondent_flag,0.985,0.924,1.050,6.401035e-01


,model_rows,mcfadden_pseudo_r2,roc_auc
0,34927,0.24,0.812


### 8.1 Turn the fitted model into an interpretable chart

After fitting the regression, this step reformats the odds ratios and confidence intervals so the strongest adjusted drivers can be communicated visually.

In [9]:
plot_table = coefficient_table.copy()
plot_table["log_odds_ratio"] = np.log(plot_table["odds_ratio"])
plot_table["log_ci_low"] = np.log(plot_table["ci_low"])
plot_table["log_ci_high"] = np.log(plot_table["ci_high"])

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(
    x=plot_table["log_odds_ratio"],
    y=plot_table["term"],
    xerr=[
        plot_table["log_odds_ratio"] - plot_table["log_ci_low"],
        plot_table["log_ci_high"] - plot_table["log_odds_ratio"],
    ],
    fmt="o",
    color="tab:blue",
    ecolor="gray",
    capsize=4,
)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Logistic model effects on food-shortage odds")
ax.set_xlabel("Log odds ratio")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 9. Vulnerability Segmentation

Clustering is kept only because it adds useful decision value after the main model. We compare `k = 3` to `k = 6`, then use a 4-cluster solution because it keeps two distinct non-irrigated risk profiles separate instead of collapsing them together.

How to explain it:

> The clusters are not magic categories. They are a dashboard-friendly way to summarize common household profiles using the strongest validated variables.

In [10]:
seg_df = df[
    [
        "food_shortage_flag",
        "household_size_derived",
        "crop_diversity_count",
        "land_cultivated_ha",
        "land_productivity_kg_per_ha",
        "country_income_rank_pct",
        "irrigated_flag",
    ]
].dropna().copy()

seg_df = seg_df[
    (seg_df["land_cultivated_ha"] > 0) & (seg_df["land_productivity_kg_per_ha"] > 0)
].copy()
seg_df["log_land_cultivated"] = np.log1p(seg_df["land_cultivated_ha"])
seg_df["log_land_productivity"] = np.log1p(seg_df["land_productivity_kg_per_ha"])

cluster_features = [
    "household_size_derived",
    "crop_diversity_count",
    "log_land_cultivated",
    "log_land_productivity",
    "country_income_rank_pct",
    "irrigated_flag",
]
scaler = StandardScaler()
X = scaler.fit_transform(seg_df[cluster_features])

sample_idx = seg_df.sample(n=min(8000, len(seg_df)), random_state=42).index
sample_positions = seg_df.index.get_indexer(sample_idx)
X_sample = X[sample_positions]

silhouette_rows = []
labels_by_k = {}
for k in range(3, 7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X)
    labels_by_k[k] = labels
    sample_labels = labels[sample_positions]
    silhouette_rows.append({"k": k, "silhouette": round(float(silhouette_score(X_sample, sample_labels)), 4)})

silhouette_df = pd.DataFrame(silhouette_rows)
display(silhouette_df)

chosen_k = 4
seg_df["cluster"] = labels_by_k[chosen_k]

,k,silhouette
0,3,0.2364
1,4,0.2209
2,5,0.2223
3,6,0.2165


### 9.1 Convert clusters into named vulnerability profiles

The raw cluster labels are not useful on their own, so this block summarizes each cluster and assigns an interpretable profile name for dashboard use.

In [11]:
cluster_summary = seg_df.groupby("cluster").agg(
    households=("cluster", "size"),
    food_shortage_rate=("food_shortage_flag", "mean"),
    household_size=("household_size_derived", "median"),
    crop_diversity=("crop_diversity_count", "median"),
    land_ha=("land_cultivated_ha", "median"),
    prod_kg_ha=("land_productivity_kg_per_ha", "median"),
    income_rank=("country_income_rank_pct", "median"),
    irrigated_share=("irrigated_flag", "mean"),
).round(3)
cluster_summary["food_shortage_rate"] = (cluster_summary["food_shortage_rate"] * 100).round(1)
cluster_summary["irrigated_share"] = (cluster_summary["irrigated_share"] * 100).round(1)

def assign_profile_name(row) -> str:
    if row["irrigated_share"] >= 80:
        return "Irrigated higher-productivity"
    if row["food_shortage_rate"] >= 75 and row["income_rank"] <= 0.35 and row["prod_kg_ha"] < 500:
        return "Low-income low-productivity rainfed"
    if row["household_size"] >= 10 and row["land_ha"] >= 5:
        return "Large-household extensive low-productivity"
    return "Diversified rainfed better-off"

cluster_summary["profile_name"] = cluster_summary.apply(assign_profile_name, axis=1)
cluster_summary = cluster_summary.reset_index().sort_values("food_shortage_rate", ascending=False)
display(cluster_summary)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=cluster_summary, y="profile_name", x="food_shortage_rate", ax=ax, orient="h")
ax.set_title("Food-shortage rate by vulnerability profile")
ax.set_xlabel("Food shortage rate (%)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

,cluster,households,food_shortage_rate,household_size,crop_diversity,land_ha,prod_kg_ha,income_rank,irrigated_share,profile_name
2,2,14330,78.7,6.0,2.0,1.500,316.667,0.242,0.3,Low-income low-productivity rainfed
1,1,6860,72.3,13.0,2.0,8.000,230.316,0.683,4.4,Large-household extensive low-productivity
0,0,12855,62.7,5.0,3.0,1.295,1300.000,0.696,0.0,Diversified rainfed better-off
3,3,7753,55.8,6.0,3.0,1.214,1375.000,0.601,100.0,Irrigated higher-productivity


## 10. Analysis Handoff for Tableau

This is not the final dashboard metric framework. It is a statistical handoff: these are the fields with enough evidence to be considered in the dashboard design.

In [12]:
tableau_driver_recommendations = pd.DataFrame(
    [
        {
            "field": "food_shortage_rate",
            "role": "Candidate headline metric",
            "why_it_stays": "Best full-dataset vulnerability outcome with broad coverage",
        },
        {
            "field": "land_productivity_kg_per_ha",
            "role": "Primary structural driver",
            "why_it_stays": "Strong bivariate and multivariable signal",
        },
        {
            "field": "irrigated_flag",
            "role": "Primary intervention variable",
            "why_it_stays": "Large descriptive gap and significant adjusted association",
        },
        {
            "field": "low_education_flag",
            "role": "Primary human-capital driver",
            "why_it_stays": "Very strong univariate and multivariable signal",
        },
        {
            "field": "country_income_rank_pct",
            "role": "Normalized welfare-position metric",
            "why_it_stays": "Built from PPP-normalized income, then ranked within country",
        },
        {
            "field": "vulnerability_profile_k4",
            "role": "Executive segmentation dimension",
            "why_it_stays": "Creates interpretable profiles for drill-down",
        },
        {
            "field": "crop_diversity_count",
            "role": "Secondary context variable",
            "why_it_stays": "Descriptively useful, but weak after controls",
        },
        {
            "field": "offfarm_flag",
            "role": "Secondary context variable",
            "why_it_stays": "Useful descriptively, but not a dominant adjusted driver",
        },
    ]
)
display(tableau_driver_recommendations)

,field,role,why_it_stays
0,food_shortage_rate,Candidate headline metric,Best full-dataset vulnerability outcome with broad coverage
1,land_productivity_kg_per_ha,Primary structural driver,Strong bivariate and multivariable signal
2,irrigated_flag,Primary intervention variable,Large descriptive gap and significant adjusted association
3,low_education_flag,Primary human-capital driver,Very strong univariate and multivariable signal
4,country_income_rank_pct,Normalized welfare-position metric,"Built from PPP-normalized income, then ranked within country"
5,vulnerability_profile_k4,Executive segmentation dimension,Creates interpretable profiles for drill-down
6,crop_diversity_count,Secondary context variable,"Descriptively useful, but weak after controls"
7,offfarm_flag,Secondary context variable,"Useful descriptively, but not a dominant adjusted driver"


## 11. Verified Statistical Takeaways

This final cell prints the strongest non-generic findings that should guide notebook 05 and Tableau design.

In [13]:
strongest_term = coefficient_table.iloc[0]
weakest_term = coefficient_table.sort_values("p_value", ascending=False).iloc[0]
top_profile = cluster_summary.iloc[0]
safest_profile = cluster_summary.iloc[-1]

findings = [
    f"Productivity remains one of the strongest adjusted drivers: each increase in log land productivity changes food-shortage odds with an odds ratio of {coefficient_table.loc[coefficient_table['term'] == 'log_land_productivity', 'odds_ratio'].iloc[0]:.3f}.",
    f"Larger cultivated land area is protective after controls: log cultivated land has an odds ratio of {coefficient_table.loc[coefficient_table['term'] == 'log_land_cultivated', 'odds_ratio'].iloc[0]:.3f}.",
    f"Low education is a major vulnerability marker: low-education households show an adjusted odds ratio of {coefficient_table.loc[coefficient_table['term'] == 'low_education_flag', 'odds_ratio'].iloc[0]:.3f}.",
    f"Official PPP-normalized income position is meaningful: country income rank has an adjusted odds ratio of {coefficient_table.loc[coefficient_table['term'] == 'country_income_rank_pct', 'odds_ratio'].iloc[0]:.3f}.",
    f"Crop diversity looks interesting in EDA but weakens after controls: its adjusted p-value is {coefficient_table.loc[coefficient_table['term'] == 'crop_diversity_count', 'p_value'].iloc[0]:.6f}.",
    f"The logistic model is useful rather than decorative: McFadden pseudo R² = {model_quality['mcfadden_pseudo_r2'].iloc[0]:.3f}, ROC AUC = {model_quality['roc_auc'].iloc[0]:.3f}.",
    f"Highest-risk profile: {top_profile['profile_name']} at {top_profile['food_shortage_rate']:.1f}% food shortage.",
    f"Lowest-risk profile: {safest_profile['profile_name']} at {safest_profile['food_shortage_rate']:.1f}% food shortage.",
    f"Strongest formal association by p-value: {strongest_term['term']} (p = {strongest_term['p_value']:.6f}).",
    f"Weakest retained core term in the full model: {weakest_term['term']} (p = {weakest_term['p_value']:.6f}).",
]

for idx, finding in enumerate(findings, start=1):
    print(f"{idx}. {finding}")

1. Productivity remains one of the strongest adjusted drivers: each increase in log land productivity changes food-shortage odds with an odds ratio of 0.828.
2. Larger cultivated land area is protective after controls: log cultivated land has an odds ratio of 0.671.
3. Low education is a major vulnerability marker: low-education households show an adjusted odds ratio of 1.268.
4. Official PPP-normalized income position is meaningful: country income rank has an adjusted odds ratio of 0.456.
5. Crop diversity looks interesting in EDA but weakens after controls: its adjusted p-value is 0.016493.
6. The logistic model is useful rather than decorative: McFadden pseudo R² = 0.240, ROC AUC = 0.812.
7. Highest-risk profile: Low-income low-productivity rainfed at 78.7% food shortage.
8. Lowest-risk profile: Irrigated higher-productivity at 55.8% food shortage.
9. Strongest formal association by p-value: log_land_productivity (p = 0.000000).
10. Weakest retained core term in the full model: fema